# Ashworth et al. (2025), evoked activity, on dandiset 001708

**Paper.** Ashworth, Martenson, Shi, De Preter, Heinricher and Mancini, "Multi-timescale dynamics
organize descending pain modulation", bioRxiv doi:10.64898/2025.12.15.693905, version 4 (June 4, 2026),
with its Supplementary Information; the claims file (`claims/001708/10.64898-2025.12.15.693905.yaml`)
was written from this text. Its code, cited for method details: github.com/nox-lab/multi-timescale-rhythmic-dynamics-in-rvm-neurons
at commit e12f3ee771f4b5506e5db0bac09762ba6044a4d6 ("the code").

**Dataset.** DANDI:001708, version 0.260604.1023, streamed from the archive. This is the version the
paper cites (its reference 72, "DANDI:001708/0.260604.1023") and the only published version; the
draft has the same 101 files and size.

This notebook tests the composition of the dataset and the trial-based (evoked) claims. The
ongoing-activity claims are in `ongoing.ipynb` (spectra, CV, period scan, coherence) and
`gp.ipynb` and `gp_methods.ipynb` (the Poisson Gaussian-process fits).

| claim | tested here |
|---|---|
| dataset-composition | yes |
| fig1c-psth-counts | yes |
| fig2-trial-counts | yes |
| figs3-neutral-unmodulated | yes |
| figs8-null-model-worse | yes |
| fig2g-switch-times | yes |
| fig2g-on-double-exponential | yes |
| fig2c-on-time-constants | yes |
| fig2h-off-linear-exponential | yes |
| fig2d-off-parameters | yes |
| tables9-s10-sigmoid-over-erf | yes |
| figs1-departure-timing | yes |
| figs1-completion-asymmetry | yes |
| recovery-timing | yes |
| tables3-s7-withdrawal-steeper | yes |
| fig1c-withdrawal-locked | yes |
| methods-withdrawal-latency | yes (latency; temperature is not in the dataset) |
| methods-heart-rate | yes |

No claim of this notebook is blocked.

In [ ]:
import sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import arviz as az
from scipy import stats
import rvm

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("dandiset", rvm.DANDISET, "version", rvm.VERSION)
T0 = time.time()

## Data check

In [ ]:
meta = rvm.dandiset_metadata()
assets = rvm.list_assets()
print(meta["name"])
print("assets:", len(assets), " subjects (archive summary):", meta["assetsSummary"]["numberOfSubjects"])
print("related:", [(r["name"], r["url"]) for r in meta.get("relatedResource", [])])
sessions, cells = rvm.load_all(assets)
print(f"loaded {len(sessions)} files in {time.time() - T0:.0f} s")
files = pd.DataFrame([{k: s[k] for k in ["path", "subject", "sex", "weight", "protocol", "identifier", "session_start"]}
                      | {"n_units": len(s["trains"]), "t_stop": max(s["t_stop"]),
                         "n_heat": len(rvm.heat(s)), "n_withdrawal": len(rvm.withdrawals(s)),
                         "n_pinch": len(rvm.pinches(s)), "n_ekg": len(rvm.ekg(s)),
                         "events": ",".join(k for k in s["events"] if not k.endswith("_labels"))} for s in sessions])
print(files.groupby("protocol").agg(files=("path", "size"), subjects=("subject", "nunique"),
                                    units=("n_units", "sum"), min_dur=("t_stop", "min"), max_dur=("t_stop", "max"),
                                    files_without_ekg=("n_ekg", lambda x: int((x == 0).sum())),
                                    files_with_pinch=("n_pinch", lambda x: int((x > 0).sum()))))
print("\nunits by protocol and class label as written in the files:")
print(pd.crosstab(cells.protocol, cells.label))
print("\nunits by protocol and class (NEUT and Neutral are the same label):")
print(pd.crosstab(cells.protocol, cells.cls, margins=True))
print("\nsubjects by protocol:", files.groupby("protocol").subject.nunique().to_dict(),
      " total subjects:", files.subject.nunique())
print("sessions per subject (ongoing):", files[files.protocol == "ongoing"].groupby("subject").size().value_counts().to_dict())
w = files.drop_duplicates("subject").weight.str.replace(" kg", "").astype(float)
print("sex:", files.drop_duplicates("subject").sex.value_counts().to_dict(),
      f" weight {1000 * w.min():.0f} to {1000 * w.max():.0f} g, mean {1000 * w.mean():.0f} +- {1000 * w.std():.0f} g"
      " (Methods: 77 male, 6 female, 191-348 g, 264 +- 27 g)")
first = files.drop_duplicates("protocol")
print("experiment descriptions:")
for i, r in first.iterrows():
    print(" ", r.protocol, ":", sessions[i]["experiment_description"])

In [ ]:
# Units, spike times and events: sorted, inside the observation interval, one interval per unit
bad_sorted = sum(int(np.any(np.diff(t) < 0)) for s in sessions for t in s["trains"])
outside = sum(int(np.sum((t < a) | (t > b))) for s in sessions for t, a, b in zip(s["trains"], s["t_start"], s["t_stop"]))
n_int = pd.Series([n for s in sessions for n in s["n_obs_intervals"]]).value_counts().to_dict()
print("units with unsorted spike times:", bad_sorted, "; spikes outside the observation interval:", outside,
      "; observation intervals per unit:", n_int)
print("observation intervals start at", sorted(set(np.round(cells.t_start, 3))), "s")
rates = cells.n_spikes / (cells.t_stop - cells.t_start)
print("mean rate per unit (Hz) by class:\n", rates.groupby(cells.cls).describe().round(2))
ev_out = []
for s in sessions:
    T = max(s["t_stop"])
    for k in ["heat", "paw_withdrawal", "pinch", "EKG"]:
        if k in s["events"]:
            e = s["events"][k]
            ev_out.append(dict(event=k, n=len(e), outside=int(np.sum((e < 0) | (e > T))), unsorted=int(np.any(np.diff(e) < 0))))
print(pd.DataFrame(ev_out).groupby("event").sum())
print("event label values:", {k: sorted({l for s in sessions for l in s["events"].get(k + "_labels", [])})
                              for k in ["heat", "paw_withdrawal", "pinch", "EKG"]})
# heat to withdrawal order: every withdrawal follows a heat onset, before the next one?
order_issues = []
for s in sessions:
    h, wd = rvm.heat(s), rvm.withdrawals(s)
    for t in wd:
        prev = h[h <= t]
        if len(prev) == 0 or t - prev[-1] > 30:
            order_issues.append((s["path"], round(t, 1)))
print("withdrawals without a heat onset in the 30 s before:", order_issues)
nh_nw = files[files.n_heat != files.n_withdrawal][["path", "n_heat", "n_withdrawal"]]
print("files whose heat and withdrawal counts differ:\n", nh_nw.to_string())

In [ ]:
# Recordings deposited more than once: identical spike trains, identical event times or identifiers
sig = {}
dups = []
for si, s in enumerate(sessions):
    for ui, t in enumerate(s["trains"]):
        key = (len(t), round(float(t[0]), 6) if len(t) else None, round(float(t[-1]), 6) if len(t) else None)
        if key in sig:
            dups.append((sig[key], (s["path"], ui)))
        sig[key] = (s["path"], ui)
print("unit pairs with the same spike count, first and last spike:", dups)
hk = {}
for s in sessions:
    k = tuple(np.round(rvm.heat(s), 3))
    hk.setdefault(k, []).append(s["path"])
print("files sharing identical heat-onset times:", [v for v in hk.values() if len(v) > 1])
print("duplicate identifiers:", files.identifier[files.identifier.duplicated()].tolist(),
      "; duplicate start times:", files.session_start[files.session_start.duplicated()].tolist())
# Units of one file share the file's events; units recorded together in one file are separate cells
same_file = cells.groupby("path").size()
print("files with more than one unit:", int((same_file > 1).sum()), "of", len(same_file))

In [ ]:
# Clock check: the spike trains and the event times share a clock. The withdrawal response of ON
# and OFF cells (rate in the 2 s after withdrawal minus the 2 s before) at the true withdrawal
# times is compared with the same contrast at times shifted by 60 s (labels shifted in time), per file.
def contrast(train, times):
    out = []
    for t in times:
        out.append(np.sum((train >= t) & (train < t + 2)) / 2 - np.sum((train >= t - 2) & (train < t)) / 2)
    return np.mean(out) if out else np.nan

rows = []
for ci, c in cells[cells.cls.isin(["ON", "OFF"])].iterrows():
    s = sessions[c.session]
    tr = s["trains"][c.unit]
    wd = rvm.withdrawals(s)
    wd = wd[(wd > 10) & (wd < c.t_stop - 70)]
    rows.append(dict(path=c.path, cls=c.cls, true=contrast(tr, wd - 0.5), shifted=contrast(tr, wd + 60 - 0.5)))
cc = pd.DataFrame(rows)
for cls, sign in [("ON", 1), ("OFF", -1)]:
    d = cc[cc.cls == cls]
    print(f"{cls}: response at withdrawal (Hz, after minus before) median true {d.true.median():.2f}, shifted by 60 s "
          f"{d.shifted.median():.2f}; cells with the expected sign: true {(sign * d.true > 0).sum()} of {len(d)}, "
          f"shifted {(sign * d.shifted > 0).sum()} of {len(d)}")
per_file = cc.assign(ok=lambda d: np.where(d.cls == "ON", d.true > 0, d.true < 0)).groupby("path").ok.any()
print("files with an ON or OFF cell responding with the expected sign at the true times:",
      int(per_file.sum()), "of", len(per_file))
print("files where the clock could not be checked this way (only NEUTRAL units):",
      sorted(set(cells.path) - set(cc.path)).__len__())

## dataset-composition

> The evoked protocol recorded 104 cells in 53 animals (35 OFF, 45 ON, 24 NEUTRAL) and the ongoing
> protocol 62 cells in 30 animals (25 OFF, 25 ON, 12 NEUTRAL).

In [ ]:
comp = cells.assign(set=cells.protocol.map({"evoked": "evoked", "evoked/ongoing": "evoked", "ongoing": "ongoing"}))
tab_a = comp.groupby("set").agg(cells=("cls", "size"), ON=("cls", lambda x: (x == "ON").sum()),
                                OFF=("cls", lambda x: (x == "OFF").sum()), NEUTRAL=("cls", lambda x: (x == "NEUTRAL").sum()),
                                animals=("subject", "nunique"), files=("path", "nunique"))
print("Reading (a), the Methods: evoked = 'evoked' and 'evoked/ongoing' files")
print(tab_a)
tab_b = comp.groupby("protocol").agg(cells=("cls", "size"), ON=("cls", lambda x: (x == "ON").sum()),
                                     OFF=("cls", lambda x: (x == "OFF").sum()), NEUTRAL=("cls", lambda x: (x == "NEUTRAL").sum()),
                                     animals=("subject", "nunique"), files=("path", "nunique"))
print("\nReading (b), Table S11's three sets (its animals: 35, 18, 50)")
print(tab_b)
exp = {"evoked": (104, 53), "ongoing": (62, 30)}
for k, (nc, na) in exp.items():
    r = tab_a.loc[k]
    print(f"{k}: cells {r.cells} (expected {nc} +- {0.1 * nc:.1f}), animals {r.animals} (expected {na} +- {0.1 * na:.1f})")

**Result: reproduced.** Under the Methods' grouping the evoked set has 104 cells (45 ON, 35 OFF, 24 NEUTRAL) in 53 animals and the ongoing set 62 cells (25, 25, 12) in 30 animals, exactly the paper's counts. Table S11's animal counts (35, 18, 50) do not match the dataset's 35, 18 and 30.

## Trial selection: fig1c-psth-counts, fig2-trial-counts, figs3-neutral-unmodulated

> fig1c-psth-counts: The population PSTHs of Figure 1c pool 70 ON and 60 OFF cells, with 185 ON and
> 155 OFF trials aligned to withdrawal and 182 ON and 151 OFF trials aligned to heat onset, for
> windows that run to 150 s.

> fig2-trial-counts: For the models, windows from 10 s before to 100 s after the event gave 195 ON
> and 170 OFF trials aligned to withdrawal and 197 ON and 172 OFF trials aligned to heat onset; cells
> were not selected by pre-trial firing and every trial with a withdrawal was included.

The rule is the one the criteria give (Spike2Tools.get_all_event_spike_trains). Pinches: the
criterion counts the next or previous pinch as an event that truncates a window. The code builds
its event list from an event named "Keyboard", while its NWB loader renames the pinch events
"keyboard" (data_loaders.rename_blocks_to_old_labels), so the code's list holds no pinch; both
are run: "with pinch" (the criterion's rule) and "without pinch" (the code as it runs).

In [ ]:
def count_table(pre, post, with_pinch, sel=cells, gap_flick=None):
    """gap_flick: how long before a withdrawal the previous event must be (default: the window start)."""
    out = {}
    for cls in ["ON", "OFF", "NEUTRAL"]:
        for ev, name in [("flick", "withdrawal"), ("heat", "heat")]:
            tr = rvm.all_trials(sessions, sel, cls, ev, pre, post, with_pinch, gap_flick if ev == "flick" else None)
            out[(cls, name)] = (len(tr), len({c for c, _ in tr}))
    return out

readings = {}
for pre, post, gap in [(-30, 150, None), (-10, 150, None), (-10, 150, 30)]:
    for wp in (True, False):
        readings[(pre, post, wp, gap)] = count_table(pre, post, wp, gap_flick=gap)
rows = []
for (pre, post, wp, gap), tab in readings.items():
    rows.append(dict(window=f"{pre} to {post} s", prev_gap_withdrawal=f"{-pre if gap is None else gap} s", pinch="with" if wp else "without",
                     ON_cells=tab[("ON", "withdrawal")][1], OFF_cells=tab[("OFF", "withdrawal")][1],
                     ON_withdrawal=tab[("ON", "withdrawal")][0], OFF_withdrawal=tab[("OFF", "withdrawal")][0],
                     ON_heat=tab[("ON", "heat")][0], OFF_heat=tab[("OFF", "heat")][0]))
fig1c = pd.DataFrame(rows)
print("expected: ON cells 70, OFF cells 60, ON/OFF withdrawal 185/155, ON/OFF heat 182/151 (10% tolerance)")
print(fig1c.to_string(index=False))
print("ON and OFF units in the dataset:", (cells.cls == "ON").sum(), (cells.cls == "OFF").sum())
expv = dict(ON_withdrawal=185, OFF_withdrawal=155, ON_heat=182, OFF_heat=151)
fig1c["all_within_10pct"] = [all(abs(r[k] - v) <= 0.1 * v for k, v in expv.items()) for _, r in fig1c.iterrows()]
fig1c["exact"] = [all(r[k] == v for k, v in expv.items()) for _, r in fig1c.iterrows()]
print(fig1c[["window", "prev_gap_withdrawal", "pinch", "all_within_10pct", "exact"]].to_string(index=False))

**Result: reproduced.** With the criterion's rule (pinches truncate windows) and the window from -30 s the counts are 180, 151, 177 and 147, within 10% of 185, 155, 182 and 151. Without pinches, as the code runs, they are exactly the paper's. With the window from -10 s and 30 s before a withdrawal for the previous event they are 180, 151, 183 and 157 with pinches and 185, 155, 188 and 161 without, all within 10%.

In [ ]:
rows = []
for wp in (True, False):
    tab = count_table(-10, 100, wp)
    rows.append(dict(pinch="with" if wp else "without",
                     ON_withdrawal=tab[("ON", "withdrawal")][0], OFF_withdrawal=tab[("OFF", "withdrawal")][0],
                     ON_heat=tab[("ON", "heat")][0], OFF_heat=tab[("OFF", "heat")][0],
                     ON_cells=tab[("ON", "withdrawal")][1], OFF_cells=tab[("OFF", "withdrawal")][1],
                     ON_cells_heat=tab[("ON", "heat")][1], OFF_cells_heat=tab[("OFF", "heat")][1]))
fig2 = pd.DataFrame(rows)
print("expected: 195, 170 (withdrawal), 197, 172 (heat), 10% tolerance")
print(fig2.to_string(index=False))
# Trial sets used by every model claim: the criterion's rule (with pinch) is the main reading;
# the code's (without pinch) is fitted as a variant for the main withdrawal-aligned models.
TR = {}
for wp in (True, False):
    for cls in ["ON", "OFF"]:
        for ev in ["flick", "heat"]:
            TR[(cls, ev, wp)] = rvm.all_trials(sessions, cells, cls, ev, -10, 100, wp)
COUNTS = {k: rvm.summed_counts(sessions, cells, v) for k, v in TR.items()}
for k, (X, Y, P) in COUNTS.items():
    print(k, "trials", P.shape[0], "bins", len(X), "total spikes", int(Y.sum()), "first five summed counts", Y[:5].astype(int))

**Result: reproduced.** From -10 to 100 s the counts are 190, 168, 192 and 169 with pinches and exactly 195, 170, 197 and 172 without, from all 70 ON and 60 OFF cells. Every model below is fitted to both trial sets.

## figs3-neutral-unmodulated

> NEUTRAL cells (n = 32; 69 withdrawal-aligned and 70 heat-aligned trials) show no modulation linked to
> heat onset or withdrawal.

The criterion takes the NEUTRAL units of the 'evoked/ongoing' and 'ongoing' files, the paper's n = 32
(Table S11: 4 NEUTRAL cells in Evoked, 20 in Evoked/Ongoing, 12 in Ongoing). This criterion was amended
after a reader's answer (questions/001708.yaml); it first took all 36 NEUTRAL units of all files. The 36
are still shown. The four NEUTRAL units of 'evoked' files are labelled "NEUT", the 32 others "Neutral";
the code selects units named exactly "Neutral" (Spike2Tools.spiketrain_iterator), the same 32.

In [ ]:
def mod_test(sel, cls, ev, pre, post, wp):
    tr = rvm.all_trials(sessions, sel, cls, ev, pre, post, wp)
    d = []
    for ci, t in tr:
        c = sel.loc[ci]
        x = sessions[c.session]["trains"][c.unit] - t
        d.append((np.sum((x >= 0) & (x < 2)) - np.sum((x >= -2) & (x < 0))) / 2.0)
    d = np.array(d)
    p = stats.wilcoxon(d).pvalue if np.any(d != 0) else 1.0
    return len(tr), len({c for c, _ in tr}), float(np.median(d)), float(np.mean(d)), float(p)

sel_crit = cells[cells.protocol.isin(["evoked/ongoing", "ongoing"])]
sel_code = cells[cells.label != "NEUT"]
sel_all = cells
print("NEUTRAL units: 'evoked/ongoing' and 'ongoing' files", int((sel_crit.cls == "NEUTRAL").sum()),
      "; named 'Neutral'", int((sel_code.cls == "NEUTRAL").sum()), "; all files", int((sel_all.cls == "NEUTRAL").sum()),
      "; the first two are the same units:",
      set(sel_crit[sel_crit.cls == "NEUTRAL"].index) == set(sel_code[sel_code.cls == "NEUTRAL"].index))
rows = []
for selname, sel in [("'evoked/ongoing' and 'ongoing' files (criterion)", sel_crit), ("all 36 NEUTRAL units", sel_all)]:
    for (pre, post) in [(-30, 150), (-10, 150)]:
        for wp in (True, False):
            for ev, evn in [("flick", "withdrawal"), ("heat", "heat")]:
                n, ncell, med, mean, p = mod_test(sel, "NEUTRAL", ev, pre, post, wp)
                rows.append(dict(selection=selname, window=f"{pre}..{post}", pinch="with" if wp else "without", event=evn,
                                 cells=ncell, trials=n, median_change_Hz=round(med, 3), mean_change_Hz=round(mean, 3), p=round(p, 4)))
neut = pd.DataFrame(rows)
print("expected: 32 cells, 69 withdrawal and 70 heat trials (10%); no change (Wilcoxon p >= 0.05)")
print(neut.to_string(index=False))
print("\npositive control, ON and OFF trials, -30..150 s window, with pinch:")
for cls in ["ON", "OFF"]:
    for ev, evn in [("flick", "withdrawal"), ("heat", "heat")]:
        n, ncell, med, mean, p = mod_test(cells, cls, ev, -30, 150, True)
        print(f"  {cls} {evn}: {n} trials, median change {med:.2f} Hz, mean {mean:.2f} Hz, p = {p:.2g}")

**Result: reproduced.** Under the criterion's selection, the NEUTRAL units of the 'evoked/ongoing' and 'ongoing' files, there are 32 cells with 67 withdrawal-aligned and 68 heat-aligned trials (window from -30 s, with pinches), within tolerance of 32, 69 and 70; without pinches the counts are exactly 69 and 70. No reading shows a NEUTRAL change at withdrawal or heat onset (median change 0 Hz, Wilcoxon p 0.24 to 0.70), while ON and OFF trials do (p = 5e-15 and 2e-16). With all 36 NEUTRAL units, the criterion before the reader's answer, the cell count (36) and the withdrawal-aligned trials (76 to 79) are outside tolerance, and there is still no change (p 0.14 to 0.65). The heat-aligned contrast has little power: on ON and OFF trials it gives p = 0.97 and 0.26, so only the withdrawal-aligned half of the test can detect a modulation.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(12, 3), sharex=True)
for ax, cls in zip(axs, ["ON", "OFF", "NEUTRAL"]):
    for ev, col in [("flick", "k"), ("heat", "tab:red")]:
        tr = rvm.all_trials(sessions, sel_crit if cls == "NEUTRAL" else cells, cls, ev, -30, 150, True)
        edges = np.arange(-30, 150.01, 1.0)
        h = np.zeros(len(edges) - 1)
        for ci, t in tr:
            c = cells.loc[ci]
            h += np.histogram(sessions[c.session]["trains"][c.unit] - t, edges)[0]
        r = h / len(tr)
        ax.plot(edges[:-1], r, color=col, lw=0.8, label=f"{'withdrawal' if ev == 'flick' else 'heat'} ({len(tr)} trials)")
        print(cls, ev, "mean rate per trial (Hz, 1 s bins) range", round(r.min(), 2), "to", round(r.max(), 2))
    ax.set_title(cls); ax.set_xlabel("time from event (s)"); ax.legend(fontsize=7)
axs[0].set_ylabel("spikes/s per trial")
plt.tight_layout(); plt.show()

## Model fits (withdrawal-aligned)

The models of the code (bayesian_cell_models.py), fitted to the summed counts of fig2-trial-counts
(50 ms bins, -10 to 100 s, 2200 bins), NUTS with 4 chains, 1000 tuning steps and 1000 draws
(Methods), target_accept 0.8 (the code's sample_model), seed fixed per fit. Every model is fitted to
both trial sets: with pinch, the criterion's rule and the main reading (no suffix), and without
pinch, the code's rule, which gives the paper's trial counts exactly (suffix `_np`). For the OFF
linear plus exponential model, the main reading puts the Methods' priors k, k2 ~ N(0, 1) on k and k2
(taken as half-normal, since the sigmoid slope and the rate k2 must be positive), and the code's
priors on log k and log k2 are a variant (`_logk`).

Where the chains start. These posteriors have local modes far below the main one: started at the
model's default point, as the code starts them, some OFF chains stay in a mode whose log posterior
is about 240 lower, and the pooled chains do not converge. Every model with a switch time is
therefore started at the highest of nine local maxima of the posterior (pymc.find_MAP from nine
switch times spread over its prior; rvm.map_start), with pymc's default jitter. This is a sampler
setting, not a method: the paper does not say how its chains were started. The two OFF linear fits
are also sampled from the default start (suffix `_default_start`), to show what it gives. For each
fit the table below gives the mean log posterior of each chain, which shows whether any chain stayed
in a lower mode.

The sampler's results depend on the machine. On one machine a run at these seeds is repeated exactly, but
on another the floating-point differences change the draws: means move in the third digit, and a fit that
converges on one machine can fail to on another (the verifier's execution had OFF_erf_linear_np unconverged,
r_hat 1.393, and the default-start fit with two chains in the lower mode). The table marks a fit as
converged when its largest r_hat is at most 1.01 and its smallest bulk ESS at least 400; conclusions use
converged fits only, and the two erf fits with linear recovery are also sampled at target_accept 0.95.


In [ ]:
FITS, DIAG = {}, []
def fit(name, builder, counts_key, seed, sample_kw=None, **kw):
    X, Y, _ = COUNTS[counts_key]
    t = time.time()
    idata = rvm.sample(builder(X, Y, **kw), seed=seed, **(sample_kw or {}))
    FITS[name] = idata
    rep = rvm.chain_report(idata)
    DIAG.append(dict(fit=name, seconds=round(time.time() - t), divergences=sum(rep["div"]), max_rhat=round(rep["rhat"], 3),
                     min_ess=round(rep["ess"]), lp_per_chain=rep["lp"],
                     lp_spread=round(max(rep["lp"]) - min(rep["lp"]), 1),
                     converged=bool(rep["rhat"] <= 1.01 and rep["ess"] >= 400)))
    return idata

SUF = {True: "", False: "_np"}  # with pinch (main reading), without pinch (the code's trial set)
for wp, suf in SUF.items():
    o = 100 * (not wp)
    fit("ON_double" + suf, rvm.build_on_double, ("ON", "flick", wp), 11 + o)
    fit("ON_single" + suf, rvm.build_on_single, ("ON", "flick", wp), 12 + o)
    fit("ON_null" + suf, rvm.build_mean, ("ON", "flick", wp), 13 + o)
    fit("OFF_linear" + suf, rvm.build_off_linear, ("OFF", "flick", wp), 21 + o, log_k=False)
    fit("OFF_linear_logk" + suf, rvm.build_off_linear, ("OFF", "flick", wp), 22 + o, log_k=True)
    fit("OFF_double" + suf, rvm.build_off_double, ("OFF", "flick", wp), 23 + o)
    fit("OFF_erf_double" + suf, rvm.build_off_erf_double, ("OFF", "flick", wp), 24 + o)
    fit("OFF_erf_linear" + suf, rvm.build_off_erf_linear, ("OFF", "flick", wp), 25 + o)
    fit("OFF_null" + suf, rvm.build_mean, ("OFF", "flick", wp), 26 + o)
# the code's start (the model's default initial point), main trial set
fit("OFF_linear_default_start", rvm.build_off_linear, ("OFF", "flick", True), 21, sample_kw={"init_map": False}, log_k=False)
fit("OFF_linear_logk_default_start", rvm.build_off_linear, ("OFF", "flick", True), 22, sample_kw={"init_map": False}, log_k=True)
# the erf model with linear recovery, sampled again with target_accept 0.95 on both trial sets (a sampler setting)
# converged: largest r_hat at most 1.01 and smallest bulk ESS at least 400
fit("OFF_erf_linear_ta95", rvm.build_off_erf_linear, ("OFF", "flick", True), 27, sample_kw={"target_accept": 0.95})
fit("OFF_erf_linear_np_ta95", rvm.build_off_erf_linear, ("OFF", "flick", False), 127, sample_kw={"target_accept": 0.95})
print(pd.DataFrame(DIAG).to_string(index=False))
print("local maxima found by map_start for the OFF linear fit (t_switch start, log posterior, t_switch at the maximum):")
print(" ", FITS["OFF_linear"].attrs["map_starts"])

In [ ]:
def elpd_table(names, ic):
    cmp = az.compare({n: FITS[n] for n in names}, ic=ic)
    return cmp[["rank", f"elpd_{ic}", "p_" + ic, "elpd_diff", "se", "dse", "warning"]]

def pair(a, b, ic):
    """ELPD of a minus that of b, and the standard error of the difference (arviz.compare)."""
    cmp = az.compare({a: FITS[a], b: FITS[b]}, ic=ic)
    sign = 1 if cmp.index[0] == a else -1
    return sign * float(cmp.loc[cmp.index[1], "elpd_diff"]), float(cmp.loc[cmp.index[1], "dse"])

## figs8-null-model-worse

> A null model with a constant mean spike count fits significantly worse than every piecewise model, for
> ON and for OFF cells, so both an evoked departure and a recovery are needed.

In [ ]:
for suf, lab in [("", "with pinch (main)"), ("_np", "without pinch")]:
    for cls, models in [("ON", ["ON_double", "ON_single"]),
                        ("OFF", ["OFF_linear", "OFF_linear_logk", "OFF_double", "OFF_erf_double", "OFF_erf_linear"])]:
        models = [m + suf for m in models]
        for ic in ["loo", "waic"]:
            print(f"--- {cls}, {ic.upper()}, {lab}")
            print(elpd_table(models + [f"{cls}_null{suf}"], ic).round(2))
            for m in models:
                d, se = pair(m, f"{cls}_null{suf}", ic)
                print(f"  {m} minus null: {d:.1f} (dse {se:.1f}); z = {d / se:.1f}; above 1 se: {d > se}; above 2 se: {d > 2 * se}")

**Result: reproduced.** With pinches the constant-mean model is below every piecewise model by 4266 to 4512 (ON) and 2983 to 2988 (OFF) in ELPD, z = 22 to 24, under LOO and WAIC; without pinches by 4229 to 4468 and 2970 to 2973. Of the withdrawal-aligned fits started at the multi-start MAP, all but one are marked converged (r_hat at most 1.007, ESS at least 716); OFF_erf_linear with pinches is not (r_hat 1.148, ESS 19), and its refit at target_accept 0.95 is (r_hat 1.004). In this execution OFF_erf_linear_np converges (r_hat 1.006), and its refit at target_accept 0.95 too (1.003). The unconverged fit is above the null by 2988 as well, so the conclusion does not rest on it.


## fig2g-on-double-exponential

> ON-cell recovery is better described by a double exponential, two decay timescales, than by a
> single exponential, which misses the sharp peak and the prolonged decay.

In [ ]:
for suf, lab in [("", "with pinch (main)"), ("_np", "without pinch")]:
    for ic in ["loo", "waic"]:
        d, se = pair("ON_double" + suf, "ON_single" + suf, ic)
        print(f"{lab}, {ic.upper()}: ELPD double {float(getattr(az, ic)(FITS['ON_double' + suf]).iloc[0]):.1f}, single "
              f"{float(getattr(az, ic)(FITS['ON_single' + suf]).iloc[0]):.1f}; double minus single {d:.1f} (dse {se:.1f})")
X, Y, _ = COUNTS[("ON", "flick", True)]
xx = X + 0.025
r2 = rvm.on_double_rate(FITS["ON_double"], xx).mean(0)
P = {k: rvm.post(FITS["ON_single"], k)[:, None] for k in ["t_switch", "log_base_rate_pre", "log_base_rate_post", "beta_1_pre", "beta_1_post", "log_k_pre", "k_post"]}
dd = xx[None, :] - P["t_switch"]
r1 = np.where(dd <= 0, np.exp(P["log_base_rate_pre"]) + np.exp(P["log_k_pre"]) * np.exp(dd * P["beta_1_pre"]),
              np.exp(P["log_base_rate_post"]) + P["k_post"] * np.exp(-dd * P["beta_1_post"])).mean(0)
pk = np.argmax(Y)
print(f"peak of the data (one 50 ms bin) {Y.max():.0f} at {X[pk]:.2f} s; double model peak {r2.max():.0f}, single {r1.max():.0f}")
for a, b in [(5, 20), (20, 60), (60, 100)]:
    m = (X >= a) & (X < b)
    print(f"  mean count {a}-{b} s: data {Y[m].mean():.1f}, double {r2[m].mean():.1f}, single {r1[m].mean():.1f}")
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(X, Y, ".", ms=1, color="0.6", label="summed counts")
ax.plot(X, r2, "k", lw=1, label="double exponential"); ax.plot(X, r1, "tab:orange", lw=1, label="single exponential")
ax.set_xlabel("time from withdrawal (s)"); ax.set_ylabel("spikes per 50 ms bin"); ax.legend(fontsize=7)
print("counts range", Y.min(), Y.max())
plt.show()

**Result: reproduced.** The double exponential is above the single exponential by 246.2 in ELPD (dse 22.7) with pinches and by 238.4 (dse 23.2) without, the paper's 238, and the single exponential misses the peak (101 against 127 for the double and 149 in the data).

## fig2g-switch-times

> The switch from response to recovery comes 0.18 s before withdrawal for ON cells (97% HDI 0.27 to
> 0.095 s before) and 1.35 s after it for OFF cells (97% HDI 0.85 to 1.9 s after), so ON cells peak
> slightly before withdrawal while OFF cells keep falling after it.

In [ ]:
def show(name, var, transform=None):
    v = rvm.post(FITS[name], var)
    if transform is not None:
        v = transform(v)
    s = rvm.summarize(v)
    return s

for name in ["ON_double", "ON_double_np"]:
    s = show(name, "t_switch")
    print(f"{name} t_switch mean {s['mean']:.3f} s, 97% HDI {s['hdi_low']:.3f} to {s['hdi_high']:.3f} "
          f"(paper -0.18, HDI -0.27 to -0.095); inside: {-0.27 <= s['mean'] <= -0.095}; below 0: {s['mean'] < 0}")
for name in ["OFF_linear", "OFF_linear_np", "OFF_linear_logk", "OFF_linear_logk_np",
             "OFF_linear_default_start", "OFF_linear_logk_default_start"]:
    s = show(name, "t_switch")
    print(f"{name} t_switch mean {s['mean']:.3f} s, 97% HDI {s['hdi_low']:.3f} to {s['hdi_high']:.3f} "
          f"(paper 1.35, HDI 0.85 to 1.9); inside: {0.85 <= s['mean'] <= 1.9}; above 0: {s['mean'] > 0}")
for name in ["OFF_linear_default_start", "OFF_linear_logk_default_start"]:
    print(f"{name}: t_switch mean per chain {FITS[name].posterior['t_switch'].mean('draw').values.round(3)}, "
          f"mean log posterior per chain {FITS[name].sample_stats['lp'].mean('draw').values.round(1)}")

**Result: reproduced.** With the Methods' priors and pinches the ON switch is at -0.191 s and the OFF switch at 1.331 s, both inside the paper's HDIs; without pinches -0.184 and 1.312 s, and with the code's log-k priors the OFF switch is at 1.369 and 1.336 s. All these fits converge. Started at the default point, the Methods-prior OFF fit does not converge (r_hat 1.69): three chains stay in a mode whose mean log posterior is about 240 lower, and its switch time is -0.636 s.

## fig2c-on-time-constants

> ON-cell firing rises before withdrawal with time constant 0.653 s and recovers with a fast component
> of 4.157 s and a slow component of 164.223 s.

In [ ]:
ref = {"beta_1_pre": ("tau_pre", 0.653, 0.570, 0.739), "beta_1_post": ("tau_fast", 4.157, 3.582, 4.722),
       "beta_slow_post": ("tau_slow", 164.223, 129.571, 199.809)}
for name in ["ON_double", "ON_double_np"]:
    n_in = 0
    for var, (lab, val, lo, hi) in ref.items():
        s = show(name, var, lambda b: 1 / b)
        inside = lo <= s["mean"] <= hi
        n_in += inside
        print(f"{name} {lab}: mean {s['mean']:.3f} s, 97% HDI {s['hdi_low']:.3f} to {s['hdi_high']:.3f}; paper {val} ({lo} to {hi}); inside: {inside}")
    print(f"  {n_in} of 3 inside the paper's HDIs")
for name in ["ON_double", "ON_double_np"]:
    print(f"{name}, 1/beta_1_post and 1/beta_slow_post, mean per chain:",
          (1 / FITS[name].posterior["beta_1_post"]).mean("draw").values.round(2),
          (1 / FITS[name].posterior["beta_slow_post"]).mean("draw").values.round(2))
print(az.summary(FITS["ON_double"], var_names=["t_switch", "log_base_rate_pre", "log_base_rate_post", "log_k_pre", "log_k_post", "k_slow"], hdi_prob=0.97).round(3))

**Result: reproduced.** tau_pre 0.656 s, tau_fast 4.231 s and tau_slow 164.8 s with pinches, and 0.653, 4.165 and 164.5 s without, all inside the paper's HDIs; the per-chain means of both recovery constants agree.

## fig2h-off-linear-exponential

> OFF-cell recovery is best described by the combined linear and exponential model; the
> double-exponential alternative fits the early recovery but fails to reach baseline, and its ELPD is
> only slightly lower.

In [ ]:
for suf, lab in [("", "with pinch (main)"), ("_np", "without pinch")]:
    for ic in ["loo", "waic"]:
        print(f"--- {ic.upper()}, OFF sigmoid models, {lab}")
        print(elpd_table(["OFF_linear" + suf, "OFF_linear_logk" + suf, "OFF_double" + suf], ic).round(2))
        for lin in ["OFF_linear", "OFF_linear_logk"]:
            d, se = pair(lin + suf, "OFF_double" + suf, ic)
            print(f"  {lin + suf} minus OFF_double{suf}: {d:.2f} (dse {se:.2f})")
    X, Y, _ = COUNTS[("OFF", "flick", suf == "")]
    rl = rvm.off_linear_rate(FITS["OFF_linear" + suf], X + 0.025).mean(0)
    pre = Y[X < -5].mean()
    late = (X >= 80)
    print(f"{lab}: OFF pre-withdrawal mean count (-10 to -5 s) {pre:.1f}; 80-100 s: data {Y[late].mean():.1f}, linear+exp model {rl[late].mean():.1f}")
    Pd = {k: rvm.post(FITS["OFF_double" + suf], k) for k in ["c", "d", "u0"]}
    print(f"{lab}: OFF double exponential asymptote c + d - u0: mean {np.mean(Pd['c'] + Pd['d'] - Pd['u0']):.1f}")

**Result: reproduced.** With the Methods' priors the linear plus exponential model has the highest ELPD of the OFF sigmoid models, 5.08 (dse 3.56) above the double exponential with pinches and 2.64 (dse 3.54) without, the paper's 2.4. The double exponential's asymptote (44.2) stays below the pre-withdrawal level (47.2).

## fig2d-off-parameters

> OFF-cell firing falls before withdrawal with sigmoid slope k = 2.852 s-1, then recovers with an
> exponential component of 4.760 s and a linear drift of 0.186 counts per second.

In [ ]:
ref = [("k", None, 2.852, 2.333, 3.392), ("k2", lambda v: 1 / v, 4.760, 3.711, 5.861), ("k3", None, 0.186, 0.174, 0.196)]
for name in ["OFF_linear", "OFF_linear_np", "OFF_linear_logk", "OFF_linear_logk_np",
             "OFF_linear_default_start", "OFF_linear_logk_default_start"]:
    n_in = 0
    for var, tf, val, lo, hi in ref:
        s = show(name, var, tf)
        inside = lo <= s["mean"] <= hi
        n_in += inside
        lab = "tau = 1/k2" if var == "k2" else var
        print(f"{name} {lab}: mean {s['mean']:.3f}, 97% HDI {s['hdi_low']:.3f} to {s['hdi_high']:.3f}; paper {val} ({lo} to {hi}); inside: {inside}")
    print(f"  {n_in} of 3 inside")
print(az.summary(FITS["OFF_linear"], var_names=["a", "b", "c", "u0", "t_switch", "k", "k2", "k3"], hdi_prob=0.97).round(3))

**Result: reproduced.** With the Methods' priors and pinches k = 2.731, tau = 4.450 s and k3 = 0.191, all inside the paper's HDIs; every other converged OFF linear fit (log-k priors, no pinches) also has 3 of 3 inside, and the fit from the default start has 0 of 3.

## tables9-s10-sigmoid-over-erf

> A Gaussian CDF (erf) response for OFF cells fits significantly worse than the sigmoid, with either
> recovery model.

In [ ]:
conv = {d["fit"]: d["converged"] for d in DIAG}
for suf, lab in [("", "with pinch (main)"), ("_np", "without pinch")]:
    for ic in ["loo", "waic"]:
        print(f"--- {ic.upper()}, {lab}")
        for sig, erf in [("OFF_double", "OFF_erf_double"), ("OFF_linear", "OFF_erf_linear"), ("OFF_linear_logk", "OFF_erf_linear"),
                         ("OFF_linear", "OFF_erf_linear_ta95"), ("OFF_linear_logk", "OFF_erf_linear_ta95")]:
            a, b = sig + suf, (erf.replace("_ta95", suf + "_ta95") if "_ta95" in erf else erf + suf)
            d, se = pair(a, b, ic)
            print(f"  {a} minus {b}: {d:.2f} (dse {se:.2f}); sigmoid higher: {d > 0}; by > 1 se: {d > se}; by > 2 se: {d > 2 * se}; "
                  f"both converged: {conv[a] and conv[b]}")


**Result: not reproduced.** The sigmoid is never above the erf model by one standard error, and with linear recovery it is below in every comparison of converged fits. With pinches: double recovery -0.08 (dse 1.40); linear recovery -0.42 (dse 1.79, WAIC; -0.40 LOO) against the converged erf fit at target_accept 0.95 (-0.61 against the unconverged fit at 0.8). Without pinches: double recovery +0.17 (dse 1.42), sigmoid higher but by less than one standard error; linear recovery -0.88 (dse 1.76) and -0.34 (dse 1.86) against the two converged erf fits. With the log-k priors the linear-recovery differences are -0.57 to -0.99. The paper's own notebook 29 also has the linear erf model higher; here, without pinches, its ELPD-LOO is -6263.89 against -6264.77 for the sigmoid (figs8 table above).


## figs1-departure-timing, figs1-completion-asymmetry, recovery-timing

> figs1-departure-timing: Both populations leave baseline well before withdrawal; OFF firing has fallen
> 10% at 1.5 s before it and ON firing has risen 10% at 1.7 s before it, and both cross 50% of their
> change within a narrow window before withdrawal (OFF 726 ms, ON 630 ms before).

> figs1-completion-asymmetry: ON firing reaches 90% of its peak 240 ms before withdrawal, while OFF
> suppression reaches 90% only 41 ms after it, so ON activation peaks before OFF suppression is complete.

> recovery-timing: After withdrawal ON cells return to 50% of their post-withdrawal baseline by 15.39 s
> but need 222 s to reach 90%; OFF cells return to 50% by 18 s and 90% by 83 s.

The rate of each posterior draw on a grid (ON 20000 points from -10 to 400 s; OFF 2000 points from
-10 to 100 s), as calculate_rate_percentages_of_peak(_OFF) of the code.

In [ ]:
def timing(name, kind, grid):
    out = {}
    rate = (rvm.on_double_rate if kind == "ON" else rvm.off_linear_rate)(FITS[name], grid)
    for pct in [0.1, 0.5, 0.9]:
        a, b = (rvm.times_on if kind == "ON" else rvm.times_off)(rate, grid, pct)
        out[pct] = (a, b)
    del rate
    return out

G_ON, G_ON100, G_OFF = np.linspace(-10, 400, 20000), np.linspace(-10, 100, 20000), np.linspace(-10, 100, 2000)
TIM = {}
for suf in ["", "_np"]:
    TIM["ON_double" + suf] = timing("ON_double" + suf, "ON", G_ON)
    TIM["ON_double_100" + suf] = timing("ON_double" + suf, "ON", G_ON100)
    for off in ["OFF_linear", "OFF_linear_logk"]:
        TIM[off + suf] = timing(off + suf, "OFF", G_OFF)
# (ON fit, OFF fit) pairs: main reading first, then the log-k priors and the code's trial set
PAIRS = [("ON_double", "OFF_linear"), ("ON_double", "OFF_linear_logk"),
         ("ON_double_np", "OFF_linear_np"), ("ON_double_np", "OFF_linear_logk_np")]
def tline(label, x, val, lo, hi):
    s = rvm.summarize(x)
    inside = lo <= s["mean"] <= hi
    print(f"{label}: mean {s['mean']:.3f} s, 97% HDI {s['hdi_low']:.3f} to {s['hdi_high']:.3f} (paper {val}, HDI {lo} to {hi}); "
          f"inside: {inside}; draws without a crossing: {s['n_nan']}")
    return inside

In [ ]:
for on, off in PAIRS:
    n = 0
    print(f"--- {on} with {off}")
    n += tline("OFF 10% fall", TIM[off][0.1][0], -1.502, -1.746, -1.306)
    n += tline("ON 10% rise", TIM[on][0.1][0], -1.7, -1.9, -1.5)
    n += tline("OFF 50% fall", TIM[off][0.5][0], -0.726, -0.81, -0.645)
    n += tline("ON 50% rise", TIM[on][0.5][0], -0.63, -0.69, -0.57)
    print(f"  {n} of 4 inside")

**Result: reproduced.** With the Methods' OFF priors and pinches OFF falls 10% at -1.533 s and 50% at -0.724 s, ON rises 10% at -1.692 s and 50% at -0.636 s; all four are inside the paper's HDIs, and so they are under the other three pairings of priors and trial sets.

In [ ]:
for on, off in PAIRS:
    print(f"--- {on} with {off}")
    a = tline("ON 90% rise", TIM[on][0.9][0], -0.24, -0.32, -0.18)
    b = tline("OFF 90% fall", TIM[off][0.9][0], 0.041, -0.095, 0.18)
    h_on, h_off = rvm.hdi(TIM[on][0.9][0]), rvm.hdi(TIM[off][0.9][0])
    rng = np.random.default_rng(5)
    x_on, x_off = TIM[on][0.9][0], rng.permutation(TIM[off][0.9][0])
    print(f"  ON HDI wholly before OFF HDI: {h_on[1] < h_off[0]} ({h_on[1]:.3f} vs {h_off[0]:.3f}); "
          f"fraction of paired draws with ON earlier: {np.nanmean(x_on < x_off):.3f}")

**Result: reproduced.** ON reaches 90% at -0.251 s and OFF at 0.072 s (Methods' priors, pinches), both inside the paper's HDIs, with disjoint HDIs and ON earlier in all paired draws; the other three pairings give the same.

In [ ]:
for on, off in PAIRS:
    n = 0
    print(f"--- {on} (grid to 400 s) with {off}")
    n += tline("ON 50% recovery", TIM[on][0.5][1], 15.39, 9.373, 22.43)
    n += tline("ON 90% recovery", TIM[on][0.1][1], 222, 183.6, 248.8)
    n += tline("OFF 50% recovery", TIM[off][0.5][1], 18, 15, 21)
    n += tline("OFF 90% recovery", TIM[off][0.1][1], 83, 82, 84)
    print(f"  {n} of 4 inside")
for suf in ["", "_np"]:
    print(f"--- variant: ON grid ending at 100 s, ON_double{suf}")
    tline("ON 50% recovery (100 s grid)", TIM["ON_double_100" + suf][0.5][1], 15.39, 9.373, 22.43)
    tline("ON 90% recovery (100 s grid)", TIM["ON_double_100" + suf][0.1][1], 222, 183.6, 248.8)

**Result: reproduced.** Recovery to 50% and 90% takes 17.0 and 224.7 s for ON (grid to 400 s) and 19.3 and 83.6 s for OFF with pinches, and 15.4, 222.0, 17.9 and 83.1 s without; all are inside the paper's HDIs. On a grid ending at 100 s the ON times are 5.8 and 67.1 s, outside.

## tables3-s7-withdrawal-steeper and fig1c-withdrawal-locked

> tables3-s7-withdrawal-steeper: Evoked transitions are sharper aligned to withdrawal than to heat onset;
> the OFF sigmoid slope is 2.852 withdrawal-aligned against 1.747 heat-aligned, and the ON rise time
> constant 0.653 s against 1.278 s.

> fig1c-withdrawal-locked: Population ON and OFF activity is sharply time-locked to paw withdrawal;
> aligned to heat onset the PSTHs show a clear lag and shallower slopes.

Heat-aligned fits use the switch-time priors of notebooks 18a (ON, U[0, 12]) and 21a (OFF, U[-4, 20]).

In [ ]:
DIAG = []
for wp, suf in SUF.items():
    o = 100 * (not wp)
    fit("ON_double_heat" + suf, rvm.build_on_double, ("ON", "heat", wp), 41 + o, p=rvm.ON_PRIORS_HEAT)
    fit("OFF_linear_heat" + suf, rvm.build_off_linear, ("OFF", "heat", wp), 42 + o, p=rvm.OFF_LIN_PRIORS_HEAT, log_k=False)
    fit("OFF_linear_logk_heat" + suf, rvm.build_off_linear, ("OFF", "heat", wp), 43 + o, p=rvm.OFF_LIN_PRIORS_HEAT, log_k=True)
print(pd.DataFrame(DIAG).to_string(index=False))
for n in ["ON_double_heat", "OFF_linear_heat", "OFF_linear_logk_heat"]:
    for suf in SUF.values():
        print(n + suf, "r_hat by variable:", {k: round(float(v), 3) for k, v in az.rhat(FITS[n + suf]).items() if v.size == 1})

In [ ]:
res_steeper = {}
for suf, lab in [("", "with pinch (main)"), ("_np", "without pinch")]:
    print(f"--- {lab}")
    for wname, hname in [("OFF_linear", "OFF_linear_heat"), ("OFF_linear_logk", "OFF_linear_logk_heat")]:
        wname, hname = wname + suf, hname + suf
        sw, sh = show(wname, "k"), show(hname, "k")
        per_chain = FITS[hname].posterior["k"].mean("draw").values.round(3)
        disjoint = sw["hdi_low"] > sh["hdi_high"]
        res_steeper[wname] = disjoint
        print(f"OFF k ({wname} / {hname}): withdrawal {sw['mean']:.3f} ({sw['hdi_low']:.3f} to {sw['hdi_high']:.3f}); heat {sh['mean']:.3f} "
              f"({sh['hdi_low']:.3f} to {sh['hdi_high']:.3f}); paper heat 1.747 (1.476 to 2.041); heat mean inside: "
              f"{1.476 <= sh['mean'] <= 2.041}; HDIs disjoint, withdrawal higher: {disjoint}; heat k mean per chain {per_chain}")
    sw, sh = show("ON_double" + suf, "beta_1_pre", lambda b: 1 / b), show("ON_double_heat" + suf, "beta_1_pre", lambda b: 1 / b)
    print(f"ON tau_pre (ON_double{suf} / ON_double_heat{suf}): withdrawal {sw['mean']:.3f} ({sw['hdi_low']:.3f} to {sw['hdi_high']:.3f}); heat {sh['mean']:.3f} "
          f"({sh['hdi_low']:.3f} to {sh['hdi_high']:.3f}); paper heat 1.278 (1.129 to 1.426); heat mean inside: "
          f"{1.129 <= sh['mean'] <= 1.426}; HDIs disjoint, withdrawal lower: {sw['hdi_high'] < sh['hdi_low']}")

**Result: reproduced.** On the same trial set, the heat-aligned OFF k (1.742 with pinches, 1.737 without) is inside the paper's HDI and its HDI is disjoint from the withdrawal-aligned one (2.731, 2.714); the heat-aligned ON tau_pre (1.305 and 1.279 s) is inside and disjoint from 0.656 and 0.653 s. All heat-aligned fits converge (r_hat at most 1.008).

In [ ]:
for suf in ["", "_np"]:
    for wname, hname, lab in [("ON_double", "ON_double_heat", "ON"), ("OFF_linear", "OFF_linear_heat", "OFF"),
                              ("OFF_linear_logk", "OFF_linear_logk_heat", "OFF (log-k priors)")]:
        sw, sh = show(wname + suf, "t_switch"), show(hname + suf, "t_switch")
        print(f"{lab}{suf} t_switch: withdrawal-aligned {sw['mean']:.3f} s ({sw['hdi_low']:.3f} to {sw['hdi_high']:.3f}); heat-aligned "
              f"{sh['mean']:.3f} s ({sh['hdi_low']:.3f} to {sh['hdi_high']:.3f}); heat HDI wholly after: {sh['hdi_low'] > sw['hdi_high']}")
print("paper: ON -0.184 vs 8.694 s (Tables S3, S5); OFF 1.346 vs 11.718 s (Tables S7, S6)")

**Result: reproduced.** The heat-aligned switch times (ON 8.704 s, OFF 11.686 s with pinches; 8.693 and 11.717 s without, the paper's 8.694 and 11.718 s) lie wholly after the withdrawal-aligned ones, and tables3-s7-withdrawal-steeper is reproduced.

## methods-withdrawal-latency

> The heat ramp was ended at paw withdrawal, with mean withdrawal latency 8.7 ± 1.1 s (mean ± SD) and
> withdrawal temperature 49.0 ± 2.0 C.

In [ ]:
lat = []
for s in sessions:
    h, wd = rvm.heat(s), rvm.withdrawals(s)
    for i, t in enumerate(h):
        nxt = h[i + 1] if i + 1 < len(h) else np.inf
        w_ = wd[(wd > t) & (wd < nxt)]
        if len(w_):
            lat.append(dict(path=s["path"], subject=s["subject"], protocol=s["protocol"], latency=w_[0] - t))
lat = pd.DataFrame(lat)
print(f"paired trials: {len(lat)} in {lat.path.nunique()} files of {lat.subject.nunique()} animals; heat onsets without a withdrawal "
      f"before the next onset: {sum(len(rvm.heat(s)) for s in sessions) - len(lat)}")
print(f"(a) all trials: mean {lat.latency.mean():.2f} s, SD {lat.latency.std():.2f} s, range {lat.latency.min():.2f} to {lat.latency.max():.2f}")
pa = lat.groupby("subject").latency.mean()
print(f"(b) per-animal means: mean {pa.mean():.2f} s, SD {pa.std():.2f} s over {len(pa)} animals")
print("by protocol:", lat.groupby("protocol").latency.agg(["mean", "std", "size"]).round(2).to_dict("index"))
print("expected 8.7 +- 1.1 s; temperature: no temperature series in the files (events:",
      sorted({k for s in sessions for k in s["events"] if not k.endswith("_labels")}), ")")

**Result: reproduced.** The latency over 334 paired trials is 8.69 +- 1.31 s (8.66 +- 1.10 s over animal means). The dataset holds no temperatures.

## methods-heart-rate

> Heart rate, from the EKG, was 391.1 ± 34.4 bpm (mean ± SD across animals).

In [ ]:
hr = []
for s in sessions:
    e = rvm.ekg(s)
    if len(e) > 10:
        ibi = np.diff(e)
        hr.append(dict(path=s["path"], subject=s["subject"], protocol=s["protocol"], bpm=60 / ibi.mean(), n=len(e),
                       ibi_min=ibi.min(), ibi_max=ibi.max(), bpm_median_ibi=60 / np.median(ibi)))
hr = pd.DataFrame(hr)
zero = sum(int(np.sum(np.diff(rvm.ekg(s)) < 0.05)) for s in sessions)
print(f"inter-beat intervals under 50 ms (duplicated beat times): {zero} of {sum(max(len(rvm.ekg(s)) - 1, 0) for s in sessions)}")
print(f"files with EKG beats: {len(hr)} of {len(sessions)} ({hr.subject.nunique()} animals); "
      f"inter-beat intervals {hr.ibi_min.min():.3f} to {hr.ibi_max.max():.3f} s")
print(f"(a) per recording: mean {hr.bpm.mean():.1f} bpm, SD {hr.bpm.std():.1f}, range {hr.bpm.min():.1f} to {hr.bpm.max():.1f}")
allb = {sub: np.concatenate([np.diff(rvm.ekg(s)) for s in sessions if s["subject"] == sub and len(rvm.ekg(s)) > 10])
        for sub in hr.subject.unique()}
pa = pd.Series({k: 60 / v.mean() for k, v in allb.items()})
print(f"(b) per animal: mean {pa.mean():.1f} bpm, SD {pa.std():.1f} over {len(pa)} animals")
print(f"from the median inter-beat interval per recording: mean {hr.bpm_median_ibi.mean():.1f} bpm, SD {hr.bpm_median_ibi.std():.1f}")
# variant: beats closer than 50 ms to the previous kept beat dropped (they cannot be heart beats at
# about 400 bpm; a cleaning step the paper does not describe)
def dedup(e, gap=0.05):
    keep = [e[0]]
    for t in e[1:]:
        if t - keep[-1] >= gap:
            keep.append(t)
    return np.array(keep)
hr["bpm_dedup"] = [60 / np.diff(dedup(rvm.ekg(sessions[files.index[files.path == p][0]]))).mean() for p in hr.path]
print(f"variant, beats under 50 ms after the previous one dropped: mean {hr.bpm_dedup.mean():.1f} bpm, SD {hr.bpm_dedup.std():.1f} per recording")
print("expected 391.1 +- 34.4 bpm")

**Result: reproduced.** 397.1 +- 34.2 bpm per recording and 395.7 +- 32.9 bpm per animal, within 391.1 +- 34.4; 20546 inter-beat intervals are under 50 ms, and without those beats the rate is 392.4 bpm.

## Summary

| claim | outcome |
|---|---|
| dataset-composition | reproduced |
| fig1c-psth-counts | reproduced |
| fig2-trial-counts | reproduced |
| figs3-neutral-unmodulated | reproduced |
| figs8-null-model-worse | reproduced |
| fig2g-on-double-exponential | reproduced |
| fig2g-switch-times | reproduced |
| fig2c-on-time-constants | reproduced |
| fig2h-off-linear-exponential | reproduced |
| fig2d-off-parameters | reproduced |
| tables9-s10-sigmoid-over-erf | not reproduced |
| figs1-departure-timing | reproduced |
| figs1-completion-asymmetry | reproduced |
| recovery-timing | reproduced |
| tables3-s7-withdrawal-steeper | reproduced |
| fig1c-withdrawal-locked | reproduced |
| methods-withdrawal-latency | reproduced |
| methods-heart-rate | reproduced |

How the analysis differs from the paper's. The chains of every model with a switch time start at the highest of
nine local maxima of the posterior (multi-start MAP), a sampler setting the paper does not give; from the code's
default start, the OFF fit with the Methods' priors does not converge. With that start every fit converges except
the erf model with linear recovery on the main trial set, which converges at target_accept 0.95; the
Methods' priors on k and k2 are the main reading, and the code's priors on log k and log k2 give the same outcomes.
Every model is fitted to the criterion's trial set (pinches truncate windows) and to the code's (without pinches),
which gives the paper's trial counts and, to within one unit, its ELPDs; both give the same outcomes.
Fits differ between machines at the same seeds (third digit of the means; convergence of some fits), so the
numbers quoted are those of this execution.

What was found about the dataset. The counts of cells, animals and trials match the paper. Four NEUTRAL units
are named "NEUT" and are left out by the paper's code; they are the NEUTRAL units of the 'evoked' files, outside
the paper's n = 32 of Figure S3. About 2% of EKG inter-beat intervals are under 50 ms, and 11
evoked files have no EKG. The pinch labels are undescribed.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.

In [ ]:
print(f"notebook runtime {(time.time() - T0) / 60:.1f} min")